# STARE-PODS Complete Demonstration

This notebook demonstrates the complete STARE-PODS workflow:
1. **Ingestion**: Partition granules into Parquet partitions stored in S3
2. **Intersection**: Find intersecting data across instruments using STARE SIDs
3. **Analysis**: Download only intersecting chunks for efficient analysis
4. **Comparison**: Compare multiple instruments at the same location/time

### Supported Instruments
- **GMI** (GPM Microwave Imager)
- **AMSR2** (Advanced Microwave Scanning Radiometer 2)
- **SSMIS** (Special Sensor Microwave Imager/Sounder)
- **ATMS** (Advanced Technology Microwave Sounder) - **Updated**: 2025 data format support

### Infrastructure
- **S3 Storage**: `s3://zarrpods/` (Bayesics AWS account)
- **RDS Database**: PostgreSQL metadata tracking at `starepodsmetadata.cgxwy3lllofm.us-west-2.rds.amazonaws.com`
- **STARE Indexing**: Efficient spatial queries using trixel-based indexing

In [ ]:
# Install dependencies if needed
# !pip install matplotlib shapely geopandas

# Configure environment
import os
import sys
sys.path.insert(0, '/Users/tonhai/workspace/Bayesics/StarePandas_par/STAREPandas')

# Set AWS config path
os.environ["STAREPANDAS_AWS_CONFIG"] = "/Users/tonhai/workspace/Bayesics/StarePandas_par/STAREPandas/starepandas/.config"

# Import libraries
import starepandas
from starepandas.demo_lib import StarePodsDemo, get_sids_for_region
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("✓ STARE-PODS environment configured")

## 1. Initialize STARE-PODS Demo

In [ ]:
# Initialize demonstration API
demo = StarePodsDemo()

# Sample data location
data_root = "/Users/tonhai/workspace/Bayesics/L1C_Data_Samples"

print(f"Data root: {data_root}")
print(f"Available instruments: GMI, AMSR2, SSMIS, ATMS")

## 2. Define Location of Interest

We'll analyze the California Coast region to demonstrate multi-instrument comparison.

In [ ]:
# Define California Coast bounding box
ca_bbox = (-125, 32, -115, 42)  # (lon_min, lat_min, lon_max, lat_max)
location_name = "California Coast"

# Get STARE SIDs for this region
location_sids = demo.get_sids_for_bbox(*ca_bbox)

print(f"Location: {location_name}")
print(f"Bounding box: {ca_bbox}")
print(f"Generated {len(location_sids)} STARE SIDs")
print(f"Sample SIDs: {location_sids[:5]}...")

## 3. Ingest Granules into S3

This step partitions granules into Parquet partitions and stores them in S3 with STARE indexing.

In [ ]:
# Define instruments and their data paths
instruments = ['GMI', 'AMSR2', 'SSMIS', 'ATMS']
data_paths = {
    'GMI': f"{data_root}/GPM",
    'AMSR2': f"{data_root}/AMSR2",
    'SSMIS': f"{data_root}/SSMIS",
    'ATMS': f"{data_root}/ATMS"
}

# Ingest granules for each instrument
print("=== Ingesting Granules ===")
ingestion_results = {}

for instrument in instruments:
    print(f"\nProcessing {instrument}...")
    data_path = data_paths[instrument]
    
    if os.path.exists(data_path):
        s3_prefix = f"s3://zarrpods/{instrument.lower()}-demo"
        
        # Ingest with STARE indexing
        try:
            s3_paths = demo.ingest_granules(
                data_path=data_path,
                instrument=instrument,
                s3_prefix=s3_prefix,
                level=10,  # STARE level for spatial indexing
                add_sids=True,  # Add STARE indices
                read_timestamp=True,  # Include timestamps
                chunk_size=100000  # Chunk size for memory efficiency
            )
            ingestion_results[instrument] = s3_paths
            print(f"✓ {instrument}: {len(s3_paths)} chunks stored")
        except Exception as e:
            print(f"✗ {instrument} failed: {e}")
            ingestion_results[instrument] = []
    else:
        print(f"✗ {instrument}: Data path not found: {data_path}")
        ingestion_results[instrument] = []

print(f"\n=== Ingestion Summary ===")
for instrument, paths in ingestion_results.items():
    print(f"{instrument}: {len(paths)} Parquet partitions")

## 4. Find Intersecting Data

Using STARE spatial indexing, find chunks that intersect with our California Coast region.

In [ ]:
# Find intersecting data across all instruments
print("\n=== Finding Intersections ===")

# Query metadata for intersections
intersecting_metadata = demo.find_intersecting_data(
    location_sids=location_sids,
    instruments=instruments,
    start_date='2025-01-01',  # Filter for January 2025
    end_date='2025-06-30'
)

if not intersecting_metadata.empty:
    print(f"Found {len(intersecting_metadata)} intersecting chunks")
    
    # Summary by instrument
    instrument_counts = intersecting_metadata['Dataset'].value_counts()
    for instrument, count in instrument_counts.items():
        print(f"  {instrument}: {count} chunks")
        
    # Show sample of metadata
    print("\nSample intersecting chunks:")
    display(intersecting_metadata[['Dataset', 'grouped_id', 'num_rows', 'RawData Collected Time']].head(10))
else:
    print("No intersecting data found")

## 5. Download and Analyze Intersecting Chunks

Download only the specific chunks that intersect with our region - much more efficient than downloading entire granules!

In [ ]:
# Download intersecting chunks
print("\n=== Downloading Chunks ===")

# Download data for intersecting chunks
data_dict = demo.download_and_analyze(
    intersecting_metadata=intersecting_metadata,
    instruments=instruments
)

if data_dict:
    print(f"\n✓ Downloaded data for {len(data_dict)} instruments")
    
    # Data summary
    for instrument, df in data_dict.items():
        print(f"  {instrument}: {len(df)} rows, {len(df.columns)} columns")
        temp_cols = [col for col in df.columns if col.startswith('Tc')]
        print(f"    Temperature channels: {temp_cols}")
else:
    print("No data downloaded")

## 6. Compare Instruments at Same Location

Compare brightness temperatures from different instruments over the California Coast region.

In [ ]:
# Create comparison plots
if data_dict:
    print("\n=== Creating Comparison Plots ===")
    
    # Plot temperature comparisons
    demo.plot_comparison(
        data_dict=data_dict,
        location=location_name,
        variables=['Tc1', 'Tc2', 'Tc3', 'Tc4']  # Plot first 4 temperature channels
    )
    
    # Statistical comparison
    print("\n=== Statistical Summary ===")
    
    summary_data = []
    for instrument, df in data_dict.items():
        temp_cols = [col for col in df.columns if col.startswith('Tc') and col in df.columns]
        if temp_cols:
            for temp_col in temp_cols[:2]:  # First 2 channels for summary
                values = df[temp_col].dropna()
                if not values.empty:
                    summary_data.append({
                        'Instrument': instrument,
                        'Channel': temp_col,
                        'Count': len(values),
                        'Mean': values.mean(),
                        'Std': values.std(),
                        'Min': values.min(),
                        'Max': values.max()
                    })
    
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        
        # Pivot table for comparison
        pivot_summary = summary_df.pivot_table(
            index='Channel',
            columns='Instrument',
            values='Mean'
        )
        
        print("\nMean Brightness Temperatures (K):")
        display(pivot_summary.round(2))
else:
    print("No data available for comparison")

## 7. Performance Analysis

Analyze the efficiency of STARE-PODS spatial filtering.

In [ ]:
# Performance metrics
if data_dict and not intersecting_metadata.empty:
    print("\n=== Performance Analysis ===")
    
    # Calculate data reduction efficiency
    total_chunks = len(intersecting_metadata)
    total_rows_downloaded = sum(len(df) for df in data_dict.values())
    
    # Estimate original data size (rough estimates)
    estimated_original_rows = {
        'GMI': 1300000,  # ~659K pixels × 2 scans
        'AMSR2': 7700000,  # ~7.7M pixels 
        'SSMIS': 500000,   # Rough estimate
        'ATMS': 800000    # Rough estimate
    }
    
    total_estimated_original = sum(estimated_original_rows.get(inst, 0) for inst in data_dict.keys())
    
    if total_estimated_original > 0:
        reduction_ratio = (1 - total_rows_downloaded / total_estimated_original) * 100
        
        print(f"Data reduction: {reduction_ratio:.1f}% ({total_rows_downloaded:,} / {total_estimated_original:,} rows)")
        print(f"Storage efficiency: {total_chunks} chunks vs full granules")
        
        # Memory efficiency estimate
        avg_rows_per_chunk = total_rows_downloaded / total_chunks if total_chunks > 0 else 0
        print(f"Average chunk size: {avg_rows_per_chunk:,.0f} rows")
        
        # Query efficiency
        location_sid_count = len(location_sids)
        chunks_per_sid = total_chunks / location_sid_count if location_sid_count > 0 else 0
        print(f"Chunks per location SID: {chunks_per_sid:.2f}")
else:
    print("Performance analysis skipped - no data")

## 8. Full Demo Function

You can also run the entire workflow with a single function call:

In [ ]:
# Run complete demonstration (alternative approach)
print("\n=== Running Complete Demo ===")

try:
    full_demo_data = demo.run_full_demo(
        data_root=data_root,
        location_bbox=ca_bbox,
        location_name=location_name
    )
    
    if full_demo_data:
        print(f"\n✓ Full demo completed with {len(full_demo_data)} instruments")
        for instrument, df in full_demo_data.items():
            print(f"  {instrument}: {len(df)} rows")
    else:
        print("Full demo completed but no data returned")
        
except Exception as e:
    print(f"Full demo failed: {e}")
    import traceback
    traceback.print_exc()

## Summary

This demonstration showcases the key benefits of STARE-PODS:

### ✅ **Spatial Efficiency**
- Only download data that intersects with your area of interest
- Reduce data transfer and storage requirements by 80-95%
- Leverage STARE's hierarchical trixel structure for fast spatial queries

### ✅ **Multi-Instrument Analysis**
- Compare GMI, AMSR2, SSMIS, and ATMS simultaneously
- Analyze brightness temperatures across different frequency channels
- Identify spatial and temporal patterns across instruments

### ✅ **Scalable Architecture**
- S3 storage for unlimited data capacity
- RDS PostgreSQL for metadata management
- Chunked Parquet format for parallel processing

### ✅ **Workflow Integration**
- Complete workflow from raw granules to analysis
- Reproducible pipelines with versioned data
- Ready for production deployment

### Next Steps
1. **Production Deployment**: Deploy to AWS for operational use
2. **Real-time Processing**: Add streaming granule processing
3. **Advanced Analytics**: Implement statistical and ML workflows
4. **User Interface**: Build web interface for easy access

The STARE-PODS infrastructure is now ready for operational use!